In [0]:
def count_check(catalog, database, operation_type, table_name, num_diff):
    df = spark.sql(f"DESCRIBE HISTORY {catalog}.{database}.{table_name}")
    df.createOrReplaceTempView("Table_count")

    # ✅ Current count
    current_row = spark.sql(f"""
        SELECT operationMetrics.numOutputRows AS cnt
        FROM Table_count
        WHERE version = (
            SELECT MAX(version)
            FROM Table_count
            WHERE LOWER(TRIM(operation)) = LOWER(TRIM('{operation_type}'))
        )
    """).first()

    final_current_count = int(current_row.cnt) if current_row and current_row.cnt else 0

    # ✅ Previous count
    previous_row = spark.sql(f"""
        SELECT operationMetrics.numOutputRows AS cnt
        FROM Table_count
        WHERE version < (
            SELECT MAX(version)
            FROM Table_count
            WHERE LOWER(TRIM(operation)) = LOWER(TRIM('{operation_type}'))
        )
        ORDER BY version DESC
        LIMIT 1
    """).first()

    final_previous_count = int(previous_row.cnt) if previous_row and previous_row.cnt else 0

    # ✅ Comparison
    if (final_current_count - final_previous_count) < int(num_diff):
        raise Exception(f'Difference is huge in {table_name}')